# AEA Segmentation on CBCT — Training Notebook

**Anterior Ethmoidal Artery Segmentation using SwinUNETR (MONAI)**

### Before you start:
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Follow the cells in order — each cell must complete before running the next
3. Training takes approximately **2–4 hours** on a Colab T4 GPU

---

## Step 1 — Verify GPU

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU')

## Step 2 — Install Dependencies

In [ ]:
# Install all required packages
# ─────────────────────────────────────────────────────────────────────────────
# MONAI is force-upgraded to 1.4.0 — 1.3.x has a uint32 overflow bug with
# Python 3.12 that crashes transform initialisation.
# The --upgrade flag is required because Colab pre-caches 1.3.2 and pip
# won't replace it without being explicitly told to upgrade.
# ─────────────────────────────────────────────────────────────────────────────

# Force upgrade MONAI first, separately, so pip resolves it correctly
!pip install -q --upgrade 'monai[all]==1.4.0'

# Install remaining packages
!pip install -q \
    pydicom \
    pynrrd \
    SimpleITK \
    nibabel \
    loguru \
    tqdm \
    scipy

# Verify
import importlib
required = {
    'monai'    : 'monai',
    'pydicom'  : 'pydicom',
    'nrrd'     : 'pynrrd',
    'SimpleITK': 'SimpleITK',
    'nibabel'  : 'nibabel',
    'loguru'   : 'loguru',
    'scipy'    : 'scipy',
}
all_ok = True
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — FAILED')
        all_ok = False

if all_ok:
    import monai
    print(f'\nAll packages ready. MONAI version: {monai.__version__}')
    if monai.__version__ < '1.4.0':
        print('WARNING: MONAI version is still < 1.4.0. Run Runtime → Restart session, then re-run this cell.')
    else:
        print('Version OK. Do NOT restart — proceed to Step 3.')
else:
    print('\nSome packages failed — run this cell again.')

## Step 3 — Get Project Code into Colab

**Choose ONE of the two options below** — run only the cell that matches your choice.

### Option A — GitHub (recommended, repo must be PUBLIC)
1. Go to [github.com](https://github.com) → New repository → set visibility to **Public** → Create
2. On your computer, open a terminal inside the `aea-segmentation` folder:
```
git init
git add .
git commit -m "AEA segmentation project"
git remote add origin https://github.com/YOUR_USERNAME/aea-segmentation.git
git push -u origin main
```
3. Replace `YOUR_GITHUB_REPO_URL` in the Option A cell below and run it

### Option B — Direct zip upload (no GitHub needed)
1. On your computer, zip the entire `aea-segmentation` folder → `aea-segmentation.zip`
2. Run the Option B cell below — it opens a file picker to upload the zip directly into Colab

In [ ]:
## OPTION A — Clone from PUBLIC GitHub repo
# Only run this cell if your repo is public.
# Private repos will fail with "could not read Username" — use Option B instead.

import os, sys
from pathlib import Path

# ── SET YOUR REPO URL HERE ─────────────────────────────────────────────────────
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/aea-segmentation.git'
# ──────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = Path('/content/aea-segmentation')

if PROJECT_DIR.exists():
    print(f'✓ Project already cloned — pulling latest changes...')
    os.system('cd /content/aea-segmentation && git pull')
else:
    ret = os.system(f'git clone {GITHUB_REPO_URL} /content/aea-segmentation')
    if ret != 0 or not PROJECT_DIR.exists():
        raise RuntimeError(
            'Git clone failed. Make sure the repository is PUBLIC.\n'
            'Go to GitHub → your repo → Settings → Change visibility → Make public\n'
            'Or use Option B (zip upload) instead.'
        )

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(str(PROJECT_DIR))
print(f'✓ Project ready at {PROJECT_DIR}')
print(f'  Files: {[f.name for f in PROJECT_DIR.iterdir() if f.is_file()]}')

In [ ]:
## OPTION B — Upload zip directly (no GitHub needed)
# Run this cell if you don't have a public GitHub repo.
# Steps:
#   1. On your computer: right-click the aea-segmentation folder → compress/zip
#   2. Run this cell — a file picker will open
#   3. Select aea-segmentation.zip — upload starts immediately

import os, sys, zipfile
from pathlib import Path
from google.colab import files as colab_files

PROJECT_DIR = Path('/content/aea-segmentation')

if PROJECT_DIR.exists():
    print('✓ Project already uploaded — skipping.')
else:
    print('Opening file picker — select aea-segmentation.zip from your computer...')
    uploaded = colab_files.upload()   # Opens browser file picker

    if not uploaded:
        raise RuntimeError('No file uploaded. Run this cell again and select the zip file.')

    zip_filename = list(uploaded.keys())[0]
    zip_path     = Path('/content') / zip_filename

    print(f'Extracting {zip_filename}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    zip_path.unlink()

    # Handle case where zip contains a subfolder vs files directly
    if not PROJECT_DIR.exists():
        # Try common naming variations
        for candidate in ['/content/aea_segmentation', '/content/aea-segmentation-main']:
            if Path(candidate).exists():
                Path(candidate).rename(PROJECT_DIR)
                break

    if not PROJECT_DIR.exists():
        raise RuntimeError(
            f'Could not find aea-segmentation folder after extraction.\n'
            f'Contents of /content/: {list(Path("/content").iterdir())}'
        )

sys.path.insert(0, str(PROJECT_DIR))
os.chdir(str(PROJECT_DIR))
print(f'✓ Project ready at {PROJECT_DIR}')
print(f'  Files: {[f.name for f in PROJECT_DIR.iterdir() if f.is_file()]}')

## Step 3c — Restore Preprocessed Data from GCS

Since the data is already preprocessed and uploaded to GCS, **skip Steps 3b download and Step 4**.
Run this cell instead — it pulls the NIfTI images and split JSONs directly from your bucket.
This takes ~1–2 minutes and replaces the 30-minute download + preprocessing step.

In [ ]:
import os, sys
from pathlib import Path

# ── SET YOUR BUCKET PATH ──────────────────────────────────────────────────────
GCS_BUCKET = "aea-checkpoints-YOURPROJECT"   # <-- same bucket name as Step 5b
GCS_PATH   = f"gs://{GCS_BUCKET}"
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR = Path("/content/aea-segmentation")
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(str(PROJECT_DIR))

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/splits",    exist_ok=True)

print("Restoring preprocessed NIfTI files from GCS...")
ret1 = os.system(f"gsutil -m cp -r {GCS_PATH}/data/processed ./data/")

print("Restoring split JSON files from GCS...")
ret2 = os.system(f"gsutil -m cp -r {GCS_PATH}/data/splits ./data/")

if ret1 == 0 and ret2 == 0:
    print("
✓ Data restored successfully. Skipping download and preprocessing.")
else:
    raise RuntimeError("GCS restore failed — check your bucket name and permissions.")

# Verify splits
import json
for split in ["train", "val", "test"]:
    path = Path(f"data/splits/{split}.json")
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        print(f"  {split:>5}: {len(data)} cases")
    else:
        print(f"  {split:>5}: NOT FOUND — check GCS path")


## Step 5 — Inspect a Sample Case

Visualise one training case to verify preprocessing is correct.

In [ ]:
import json
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

from config import SPLITS_DIR

# Load the first training case
with open(SPLITS_DIR / 'train.json') as f:
    train_manifest = json.load(f)

sample = train_manifest[0]
print(f"Case: {sample['case_id']}")

image = sitk.GetArrayFromImage(sitk.ReadImage(sample['image']))  # (Z, Y, X)
mask  = sitk.GetArrayFromImage(sitk.ReadImage(sample['mask']))

print(f'Image shape: {image.shape}, range: [{image.min():.0f}, {image.max():.0f}] HU')
print(f'Mask shape:  {mask.shape},  labels: {np.unique(mask)}')

# Find a slice containing AEA voxels
aea_slices = np.where(mask > 0)[0]
mid_slice  = aea_slices[len(aea_slices) // 2] if len(aea_slices) > 0 else image.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(image[mid_slice], cmap='gray', vmin=-1000, vmax=1000)
axes[0].set_title(f'CBCT — Slice {mid_slice} (axial)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(image[mid_slice], cmap='gray', vmin=-1000, vmax=1000)
overlay = np.ma.masked_where(mask[mid_slice] == 0, mask[mid_slice])
axes[1].imshow(overlay, cmap='bwr', alpha=0.6, vmin=0.5, vmax=2.5)
axes[1].set_title(f'AEA Overlay (Red=Left, Blue=Right) — Slice {mid_slice}', fontsize=13)
axes[1].axis('off')

plt.suptitle(f"Patient: {sample['case_id']}", fontsize=15)
plt.tight_layout()
plt.savefig('sample_case.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sample visualisation saved as sample_case.png')

## Step 5b — Set up GCS checkpoint backup (Colab Enterprise)

Run this **before starting training**. It creates a Cloud Storage bucket and tells
`train.py` where to automatically back up checkpoints.

While training runs:
- Every **10 epochs** → `swinunetr_last.pth` is copied to GCS (resume point)
- Every **new best Dice** → `swinunetr_best.pth` is copied to GCS

If Colab crashes, just download the checkpoint from GCS and resume — maximum 10 epochs lost.

In [ ]:
import os, subprocess
from pathlib import Path

# ── SET YOUR BUCKET NAME ───────────────────────────────────────────────────────
# Use your Google Cloud project ID as the bucket name for uniqueness.
# You can find your project ID in the top bar of Google Cloud Console.
GCS_BUCKET = 'aea-checkpoints-YOURPROJECT'   # <-- change this
# ──────────────────────────────────────────────────────────────────────────────

GCS_PATH = f'gs://{GCS_BUCKET}/checkpoints'

# Create the bucket if it doesn't exist yet (safe to re-run)
result = subprocess.run(
    ['gsutil', 'mb', '-l', 'EU', f'gs://{GCS_BUCKET}'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✓ Bucket created: gs://{GCS_BUCKET}')
elif 'already exists' in result.stderr:
    print(f'✓ Bucket already exists: gs://{GCS_BUCKET}')
else:
    print(f'✗ Error: {result.stderr}')
    raise RuntimeError('Could not create GCS bucket — check your project ID and billing.')

# Tell train.py where to back up checkpoints
os.environ['AEA_GCS_BACKUP'] = GCS_PATH
print(f'✓ Backup path set: {GCS_PATH}')
print()
print('train.py will now automatically copy checkpoints to GCS:')
print(f'  Best checkpoint → {GCS_PATH}/swinunetr_best.pth')
print(f'  Last checkpoint → {GCS_PATH}/swinunetr_last.pth  (every 10 epochs)')
print()
print('If Colab crashes, download the checkpoint from GCS Console or run:')
print(f'  gsutil cp {GCS_PATH}/swinunetr_last.pth /content/aea-segmentation/models/swinunetr/')

## Step 6 — Train the Model

Expected duration on Colab T4:
- With full cache (cache_rate=1.0): ~2-4 hours
- With early stopping, may finish sooner

**Important:** Colab sessions disconnect after ~12 hours of inactivity.
The last checkpoint is saved every 10 epochs, so you can resume if disconnected.

In [ ]:
import torch, os
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

total = torch.cuda.get_device_properties(0).total_memory
print(f'VRAM free: {(total - torch.cuda.memory_allocated(0)) / 1e9:.1f} GB / {total / 1e9:.1f} GB')

# --persistent  : saves processed tensors to /tmp/aea_cache (Colab SSD, ~100 GB free)
#                 instead of RAM. First run takes ~5 min to build the cache.
#                 All subsequent epochs load from disk — fast, no RAM OOM.
# --num_workers 0: single-process loading, most stable on Colab
!PYTORCH_ALLOC_CONF=expandable_segments:True python src/train.py \
    --persistent \
    --num_workers 0

In [ ]:
import torch, os
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from config import SWINUNETR_DIR, TRAIN_CONFIG
last_ckpt = str(SWINUNETR_DIR / TRAIN_CONFIG['last_model_name'])
print(f'Resuming from: {last_ckpt}')

# --persistent reuses the existing /tmp/aea_cache built during the first run
# so there's no cache rebuild cost on resume
!PYTORCH_ALLOC_CONF=expandable_segments:True python src/train.py \
    --persistent \
    --num_workers 2 \
    --resume "{last_ckpt}"

## Step 6b — Fine-tune from Best Checkpoint (run after Step 6)

Use this cell **instead of the resume cell** when you want to continue training from
`swinunetr_best.pth` with the improved loss function and augmentations.

**What changes in fine-tune mode:**
- Loads only the model weights — the optimizer is reset to avoid momentum artefacts from the previous run
- Uses a smaller learning rate (1e-5 instead of 1e-4) so already-learned features are not disrupted
- No warmup phase — starts cosine annealing from epoch 1
- Class-weighted cross-entropy loss (background × 0.1) focuses gradients on the rare AEA voxels
- Richer augmentations: Gaussian noise/blur + contrast jitter (already wired into `dataset.py`)

**Expected outcome:** Dice score should climb from 0.55 toward 0.65–0.70 over another 200–300 epochs.
Early stopping (patience = 50 validation checks = 250 real epochs) terminates automatically.

In [ ]:
import torch, os
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from config import SWINUNETR_DIR, TRAIN_CONFIG
best_ckpt = str(SWINUNETR_DIR / TRAIN_CONFIG['best_model_name'])

if not os.path.exists(best_ckpt):
    raise FileNotFoundError(
        f'Best checkpoint not found at {best_ckpt}\n'
        'Make sure Step 6 (or a previous training run) has completed first.'
    )

print(f'Fine-tuning from: {best_ckpt}')
print('Improvements vs original training:')
print('  • LR         : 1e-4 → 1e-5  (preserves learned features)')
print('  • CE loss     : class weights [0.1, 1.0, 1.0] (background downweighted)')
print('  • Augmentation: + Gaussian noise, Gaussian blur, contrast jitter')
print('  • Optimizer   : reset (fresh Adam momentum, no warmup)')
print()

# --finetune   : loads model weights only, resets optimizer at LR=1e-5
# --persistent : reuses existing /tmp/aea_cache (no cache rebuild)
# --num_workers 0: single-process, most stable on Colab
!PYTORCH_ALLOC_CONF=expandable_segments:True python src/train.py \
    --persistent \
    --num_workers 0 \
    --finetune \
    --resume "{best_ckpt}"

## Step 7 — Plot Training History

In [ ]:
import json
import matplotlib.pyplot as plt
from config import LOGS_DIR

with open(LOGS_DIR / 'training_history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Training Loss (DiceCE)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

val_epochs = list(range(4, len(history['train_loss']) + 1, 5))[:len(history['val_dice'])]
axes[1].plot(val_epochs, history['val_dice'], color='green', marker='o', markersize=3)
axes[1].set_title('Validation Dice Score', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(val_epochs, history['val_hd95'], color='crimson', marker='o', markersize=3)
axes[2].set_title('Validation HD95 (mm)', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('HD95 (mm)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('SwinUNETR Training Progress — AEA Segmentation', fontsize=14)
plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()
print('Training history plot saved as training_history.png')

## Step 8 — Evaluate on Test Set

In [ ]:
import torch
import json
from config import SWINUNETR_DIR, TRAIN_CONFIG, SPLITS_DIR, INFERENCE_CONFIG
from src.dataset import get_dataloader
from src.evaluate import evaluate_model
from src.train import build_model
from src.utils import save_json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best model
best_ckpt = SWINUNETR_DIR / TRAIN_CONFIG['best_model_name']
model = build_model(device, pretrained=False)  # Architecture only, no pretrained weights
checkpoint = torch.load(str(best_ckpt), map_location=device)
model.load_state_dict(checkpoint['model'])
print(f"Loaded best model (val Dice: {checkpoint['best_dice']:.4f})")

# Load test set
with open(SPLITS_DIR / 'test.json') as f:
    test_manifest = json.load(f)
test_loader = get_dataloader(test_manifest, mode='test', num_workers=2)

# Evaluate
print('\nRunning test set evaluation...')
test_results = evaluate_model(
    model      = model,
    dataloader = test_loader,
    device     = device,
    roi_size   = INFERENCE_CONFIG['roi_size'],
    sw_batch   = INFERENCE_CONFIG['sw_batch_size'],
)

# Save and display results
save_json(test_results, LOGS_DIR / 'test_results.json')

print('\n' + '='*50)
print('FINAL TEST SET RESULTS')
print('='*50)
print(f"{'Metric':<20} {'AEA Left':>10} {'AEA Right':>10} {'Mean':>10}")
print('-'*50)
print(f"{'Dice (DSC)':<20} {test_results['dice_aeal']:>10.4f} {test_results['dice_aear']:>10.4f} {test_results['dice_mean']:>10.4f}")
print(f"{'IoU (Jaccard)':<20} {test_results['iou_aeal']:>10.4f} {test_results['iou_aear']:>10.4f} {test_results['iou_mean']:>10.4f}")
print(f"{'HD95 (mm)':<20} {test_results['hd95_aeal']:>10.4f} {test_results['hd95_aear']:>10.4f} {test_results['hd95_mean']:>10.4f}")
print('='*50)

## Step 9 — Visualise Predictions on Test Cases

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import SimpleITK as sitk
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

from config import INFERENCE_CONFIG
from src.postprocess import full_postprocess

model.eval()
post_pred = AsDiscrete(argmax=True)

# Visualise first 3 test cases
n_show = min(3, len(test_manifest))

for i, case in enumerate(test_manifest[:n_show]):
    image_np = sitk.GetArrayFromImage(sitk.ReadImage(case['image']))
    mask_np  = sitk.GetArrayFromImage(sitk.ReadImage(case['mask']))

    # Run inference on a single case (reuse test_loader batch)
    from src.dataset import get_transforms
    from monai.data import CacheDataset, DataLoader

    single_loader = DataLoader(
        CacheDataset([case], transform=get_transforms('test'), cache_rate=1.0),
        batch_size=1
    )
    batch = next(iter(single_loader))
    with torch.no_grad():
        logits = sliding_window_inference(
            batch['image'].to(device),
            roi_size      = INFERENCE_CONFIG['roi_size'],
            sw_batch_size = INFERENCE_CONFIG['sw_batch_size'],
            predictor     = model,
            overlap       = 0.5,
        )
    pred = post_pred(logits[0]).cpu().numpy().squeeze()  # (H, W, D)
    pred = full_postprocess(pred)

    # Find best slice
    aea_z = np.where((pred > 0) | (mask_np > 0))[0]
    z_mid = aea_z[len(aea_z) // 2] if len(aea_z) > 0 else image_np.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    axes[0].set_title('CBCT', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    gt_overlay = np.ma.masked_where(mask_np[z_mid] == 0, mask_np[z_mid])
    axes[1].imshow(gt_overlay, cmap='bwr', alpha=0.7, vmin=0.5, vmax=2.5)
    axes[1].set_title('Ground Truth', fontsize=11)
    axes[1].axis('off')

    axes[2].imshow(image_np[z_mid], cmap='gray', vmin=-500, vmax=1500)
    pred_overlay = np.ma.masked_where(pred[z_mid] == 0, pred[z_mid])
    axes[2].imshow(pred_overlay, cmap='bwr', alpha=0.7, vmin=0.5, vmax=2.5)
    axes[2].set_title('Prediction (post-processed)', fontsize=11)
    axes[2].axis('off')

    plt.suptitle(f"Test Case: {case['case_id']} — Slice {z_mid}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"prediction_{case['case_id']}.png", dpi=120, bbox_inches='tight')
    plt.show()

print(f'Visualisation complete for {n_show} test cases.')

## Step 10 — Download Trained Model to Your Computer

Downloads the best model checkpoint and logs directly to your laptop.
The file will appear in your browser's default downloads folder.

In [ ]:
import shutil, zipfile
from pathlib import Path
from google.colab import files
from config import SWINUNETR_DIR, LOGS_DIR, TRAIN_CONFIG

# Bundle model + logs into a single zip for download
bundle_path = Path('/content/aea_model_bundle.zip')

with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    best_ckpt = SWINUNETR_DIR / TRAIN_CONFIG['best_model_name']
    if best_ckpt.exists():
        zf.write(best_ckpt, f'models/swinunetr/{best_ckpt.name}')
        print(f'✓ Added model checkpoint ({best_ckpt.stat().st_size / 1e6:.0f} MB)')
    else:
        print('✗ Best checkpoint not found — did training complete?')

    # Training history + test results
    for log_file in ['training_history.json', 'test_results.json']:
        p = LOGS_DIR / log_file
        if p.exists():
            zf.write(p, f'logs/{log_file}')
            print(f'✓ Added {log_file}')

bundle_size = bundle_path.stat().st_size / 1e6
print(f'\nBundle ready: {bundle_path.name} ({bundle_size:.0f} MB)')
print('Starting download to your computer...')

# This triggers a browser download
files.download(str(bundle_path))

print('\nDone! After download:')
print('  1. Extract aea_model_bundle.zip')
print('  2. Copy models/swinunetr/swinunetr_best.pth into your local aea-segmentation/models/swinunetr/')
print('  3. Run: python run.py')

---
## Training Complete!

Your trained model is saved in `models/swinunetr/swinunetr_best.pth`.

**Next steps:**
- Download the `models/swinunetr/` folder back to your local machine
- Place it inside the `aea-segmentation/` project folder
- Run the web UI: `python ui/app.py`

The agent will automatically load the trained model for inference.